In [ ]:
from pyrosetta import *
from pyrosetta import AtomID, dihedral_degrees
from pyrosetta.rosetta.core.scoring import dslf_fa13


def analyze_disulfide_torsions(pose):
    """
    Extract disulfide torsion angles (χ1, χ2, χ3, χ2', χ1') and estimate dslf_fa13 energy.

    :param pose: Pose with disulfides detected
    :return: List of dictionaries with angles and energy in kcal/mol
    """
    results = []

    # Ensure disulfide bonds are detected
    pose.conformation().detect_disulfides()

    # Scoring function with disulfide torsion term
    scorefxn = get_fa_scorefxn()
    scorefxn.set_weight(dslf_fa13, 1.0)

    # Total dslf_fa13 energy with all disulfides
    scorefxn(pose)
    full_dslf = pose.energies().total_energies()[dslf_fa13]

    seen = set()
    for i in range(1, pose.total_residue() + 1):
        if pose.is_disulfide_bonded(i):
            j = pose.disulfide_partner(i)
            if (j, i) in seen:
                continue
            seen.add((i, j))

            # Extract χ angles
            chi1_i = pose.chi(1, i)
            chi2_i = pose.chi(2, i) if pose.residue(i).nchi() >= 2 else None
            chi1_j = pose.chi(1, j)
            chi2_j = pose.chi(2, j) if pose.residue(j).nchi() >= 2 else None

            # χ3: dihedral CB–SG–SG'–CB'
            rsd_i = pose.residue(i)
            rsd_j = pose.residue(j)
            chi3 = dihedral_degrees(
                AtomID(rsd_i.atom_index("CB"), i),
                AtomID(rsd_i.atom_index("SG"), i),
                AtomID(rsd_j.atom_index("SG"), j),
                AtomID(rsd_j.atom_index("CB"), j),
            )

            # Estimate torsional energy by deletion
            test_pose = Pose()
            test_pose.assign(pose)
            test_pose.conformation().delete_disulfide(i)
            scorefxn(test_pose)
            broken_dslf = test_pose.energies().total_energies()[dslf_fa13]
            bond_energy = full_dslf - broken_dslf

            results.append({
                "res1": i,
                "res2": j,
                "chi1": chi1_i,
                "chi2": chi2_i,
                "chi3": chi3,
                "chi2'": chi2_j,
                "chi1'": chi1_j,
                "dslf_fa13_energy": bond_energy,
            })

    return results


In [ ]:
import math

import numpy as np
from scipy.optimize import minimize


def calculate_dse(x):
    """
    Calculate Disulfide Energy (DSE) in kJ/mol based on chi angles.

    Returns:
    float: DSE value in kJ/mol
    """
    chi1, chi2, chi3, chi4, chi5 = x
    dse = (
        8.37 * (1 + math.cos(3 * chi1))
        + 8.37 * (1 + math.cos(3 * chi5))
        + 4.18 * (1 + math.cos(3 * chi2))
        + 4.18 * (1 + math.cos(3 * chi4))
        + 14.64 * (1 + math.cos(2 * chi3))
        + 2.51 * (1 + math.cos(3 * chi3))
    )

    return dse


def energy_function(x):
    chi1, chi2, chi3, chi4, chi5 = x
    energy = 2.0 * (np.cos(np.deg2rad(3.0 * chi1)) + np.cos(np.deg2rad(3.0 * chi5)))
    energy += np.cos(np.deg2rad(3.0 * chi2)) + np.cos(np.deg2rad(3.0 * chi4))
    energy += (
        3.5 * np.cos(np.deg2rad(2.0 * chi3))
        + 0.6 * np.cos(np.deg2rad(3.0 * chi3))
        + 10.1
    )
    return energy


initial_guess = [
    -60.0,
    -60.0,
    90.0,
    -60.0,
    -60.0,
]  # initial guess for chi1, chi2, chi3, chi4, chi5
result_egs = minimize(energy_function, initial_guess, method="Nelder-Mead")
minimum_energy_egs = result_egs.fun
inputs_egs = result_egs.x
dse_value_egs = energy_function(inputs_egs)
dse_value_egs

result_kj = minimize(calculate_dse, initial_guess, method="Nelder-Mead")
minimum_energy_kj = result_kj.fun
inputs_kj = result_kj.x
dse_value = calculate_dse(inputs_kj)
dse_value

# Display the results
print("Minimum energy (kJ/mol):", minimum_energy_kj)
print("Chi angles (degrees):", inputs_kj)
print("DSE value (kcal/mol):", dse_value)

# Display the results
print("Minimum energy (kcal/mol):", minimum_energy_egs)
print("Chi angles (degrees):", inputs_egs)
print("DSE value (kJ/mol):", dse_value_egs)